In [1]:
import xarray as xr
import rasterio
from rasterio.transform import from_origin
import numpy as np

In [2]:
from rasterio.crs import CRS

In [3]:
import os

In [4]:
import netCDF4
import rioxarray
import rasterio

In [5]:
base = os.path.join(os.getcwd(),'..')

In [6]:
data = os.path.join(base,'data(LPJmL)','new_run')

In [7]:
for r,d,f in os.walk(data):
    for fl in f:
        if fl.endswith('.nc') and 'cleaf' in fl:
            print(fl)
            fn = os.path.join(r,fl)


In [8]:
ls = ("temperate_cereals","rice","maize","tropical cereals","pulses","temperate roots",
             "tropical roots","sunflower","soybeans","groundnuts","rapeseed","sugarcane", 
      "barley","cotton","wheat2","rice2", "rice3","others","grasses","biofuels1","biofuels2")


In [9]:

# ds = netCDF4.Dataset(fn)
ds = xr.open_dataset(fn, decode_times=False)

In [10]:
ds

In [13]:
var_name = 'C_leaf_pft'
ds = xr.open_dataset(fn, decode_times=False, engine='netcdf4')
da = ds[var_name]


evap = ds[var_name]    
pft_names = ds["NamePFT"].values  # strings

In [16]:
c=0
for d in pft_names:
    if 'maize' in d:
        print(d,c)
    c+=1

rainfed maize 13
irrigated maize 34


In [14]:

nc_file = fn        
os.makedirs(os.path.join(base,'tiffs',var_name),exist_ok = True)
nodata = -1e+32

ds = xr.open_dataset(nc_file, decode_times=False, engine='netcdf4')
da = ds[var_name]


evap = ds[var_name]    
pft_names = ds["NamePFT"].values  # strings

ntime, bands, nlat, nlon = da.shape

lat = da["lat"].values
lon = da["lon"].values

da = da.squeeze()
da = da.sortby("lat", ascending=True)

dlat = abs(lat[1] - lat[0])
dlon = abs(lon[1] - lon[0])


transform = from_origin(
    lon.min()-dlon/2,
    lat.max()+dlat/2,
    dlon,
    dlat
)


fill_value = da.attrs.get("_FillValue", nodata)


for t in range(evap.sizes["time"]):
    time_val = ds["time"].values[t]

    out_tif = os.path.join(base,'tiffs',var_name,var_name+f"_time_{t:03d}.tif")
    
    data = evap.isel(time=t).values.astype(np.float32)

    with rasterio.open(
        out_tif,
        "w",
        driver="GTiff",
        height=nlat,
        width=nlon,
        count=bands,               
        dtype="float32",
        crs="""GEOGCS["WGS 84",
            DATUM["WGS_1984",
                SPHEROID["WGS 84",6378137,298.257223563]],
            PRIMEM["Greenwich",0],
            UNIT["degree",0.0174532925199433]]""",
        transform=transform,
        nodata=nodata,
        compress="lzw"
    ) as dst:
    
    
        for i, pft_name in enumerate(pft_names):
            band = data[i, :, :]
            dst.write(band, i + 1)
            dst.set_band_description(i + 1, str(pft_name))
            
    with rasterio.open(out_tif) as src:
        data = src.read()      
        profile = src.profile

    data_flipped = np.flip(data, axis=1)
    
    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(data_flipped)


    print(f"Written {out_tif}")


Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_000.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_001.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_002.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_003.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_004.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_005.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_006.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_007.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_008.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_009.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_010.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_011.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_012.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_013.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_014.tif
Written notebooks\..\tiffs\C_leaf_pft\C_leaf_pft_time_015.tif
Written 